# Building ESCM, one component at a time

**An executable guide to the refactored Event-Driven Spiking Cortical Model (ESCM).**

ESCM aims to maintain a persistent neural representation of an event-camera scene. Unlike a conventional camera, an event camera reports local brightness **changes**, rather than complete image frames. The model will eventually turn these asynchronous events into a persistent *output spike stream*.

This notebook is the entry point for the **new implementation**, not a reproduction of the historical experiments. We will extend it as we agree on each stage's scientific and numerical behavior.

### Pipeline roadmap

| Stage | Purpose | Status |
|:---|:---|:---|
| **1. Decode events** | Read YARP recordings into timestamped pixel events | **Implemented below** |
| 2. Model input adapter | Map events to the reference simulator's time and input representation | Planned |
| 3. Neuron dynamics | Integrate membrane and input states | Planned |
| 4. Persistence | Define how activity persists and switches off | Planned |
| 5. Recurrent interactions | Specify excitation, neighborhood, and inhibition | Planned |
| 6. Execution and output | Simulate, record output spikes, and evaluate | Planned |

**Design principle:** decoding data must not implicitly make decisions about neuronal dynamics, timestep, or interpretation of polarity.

## 0. Set up the development environment

Run these commands **in a terminal**, from the repository root (not in a notebook cell):

```bash
mkvirtualenv -p python3.12 event-scm
uv pip install -e '.[dev]'
uv pip install ipykernel matplotlib
python -m ipykernel install --user --name event-scm --display-name "Python (event-scm)"
```

If the environment already exists, run `workon event-scm` instead of `mkvirtualenv`. `pip install -e '.[dev]'` is also supported.

Choose the **Python (event-scm)** kernel in Jupyter or VS Code. The repository is installed in editable mode, so updates to `src/escm/` become available without reinstalling.

This notebook assumes it is stored under `notebooks/` and the recording data has been cloned with the repository. The legacy Brian2 scripts have **not** been migrated into the new Python package.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from escm.event_input import load_events

# Resolve the checkout root regardless of whether Jupyter starts in the
# repository root or in its notebooks/ directory.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the Event-SCM-Refactor checkout.")

print("Python:", sys.version.split()[0])
print("Repository:", ROOT)
print("NumPy:", np.__version__)

## 1. Decode the event-camera recording

### What is an event?

A DVS event typically provides four fields:

- `ts`: timestamp; **seconds** after decoding by the configured bimvee importer
- `x`, `y`: pixel coordinates (integers)
- `pol`: polarity, indicating the sign of the detected brightness change

A recording contains a stream of such events. Multiple events can arrive at the same pixel, and several events can share a timestamp. The loader deliberately preserves these details.

### The loader's responsibilities

`escm.event_input.load_events` loads **one YARP recording directory** (for example, a directory containing `data.log` and `info.log`), explicitly selects a DVS channel, validates arrays and coordinates, and returns decoded events with source metadata.

It **does not** sort, filter, deduplicate, rescale timestamps, flatten pixels to neuron indices, or create Brian2 objects. We will decide those transformations in their respective stages.

**Timestamp warning:** `timestamp_clock_ns` is the duration of one timestamp *tick in the recording format*, not the simulation timestep. It must match the specific recording. For YARP logs without per-event timestamps, bimvee can derive microsecond timestamps from bottle times (`1000 ns` per tick). Do **not** assume that value is correct for every event log.

### 1.1 Select a recording and its metadata

The example points to the repository's **MVSEC short outdoor / left DVS** recording. Set `CHANNEL`, `WIDTH`, `HEIGHT`, and `TIMESTAMP_CLOCK_NS` according to the selected recording and its actual encoding.

The channel name here refers to bimvee's **decoded** channel (e.g. `left`), not simply the folder name. If the decoder exposes a different channel, the loader will report available ones.

In [ ]:
RECORDING = ROOT / "InputData" / "MVSEC_short_outdoor" / "leftdvs"
CHANNEL = "left"
WIDTH, HEIGHT = 346, 260

# EXAMPLE ONLY: verify the raw recording's timestamp encoding first.
TIMESTAMP_CLOCK_NS = 1000

print("Recording:", RECORDING)
print("data.log exists:", (RECORDING / "data.log").is_file())
if not RECORDING.is_dir():
    raise FileNotFoundError(f"Sample recording missing: {RECORDING}")

### 1.2 Load the events

This operation decodes the whole recording into memory. It is appropriate for a notebook example; the eventual real-time implementation will need a separate streaming interface.

If the selected channel is wrong, inspect the available channels reported in the exception. If the timestamp clock is wrong, the resulting seconds will be wrong even if decoding succeeds.

In [ ]:
recording = load_events(
    RECORDING,
    channel=CHANNEL,
    width=WIDTH,
    height=HEIGHT,
    timestamp_clock_ns=TIMESTAMP_CLOCK_NS,
)
events = recording["events"]
ts, x, y, pol = (events[key] for key in ("ts", "x", "y", "pol"))

print(f"Loaded {len(ts):,} events from {recording['channel']!r}")
print(f"Sensor: {recording['width']} × {recording['height']}")
print(f"Timestamp unit: {recording['timestamp_unit']}")
print(f"Time range: {ts[0]:.6f} to {ts[-1]:.6f} s" if len(ts) else "No events")
print("Importer metadata:", recording["info"])

### 1.3 Inspect the decoded events

Before connecting any model, verify that the data looks physically plausible.

The checks below are **diagnostics**, not preprocessing:
- events remain in nondecreasing timestamp order;
- pixel coordinates stay inside the declared sensor shape;
- both polarities are counted separately;
- consecutive events at the same time are not removed.

A realistic recording duration is a useful sanity check for the chosen timestamp tick duration. It cannot, by itself, prove that the time base is correct.

In [ ]:
assert len(ts) == len(x) == len(y) == len(pol)
assert np.all(np.diff(ts) >= 0)
assert np.all((x >= 0) & (x < WIDTH))
assert np.all((y >= 0) & (y < HEIGHT))

print("Duration (s):", float(ts[-1] - ts[0]) if len(ts) else 0.0)
print("ON / positive events:", int(np.count_nonzero(pol)))
print("OFF / negative events:", int(np.count_nonzero(~pol.astype(bool))))
print("Adjacent equal timestamps:", int(np.count_nonzero(np.diff(ts) == 0)))
print("First five (ts, x, y, pol):")
for row in zip(ts[:5], x[:5], y[:5], pol[:5]):
    print(row)

### 1.4 Visualize event activity (optional)

For orientation only, the following plot counts events per pixel in the **first 50 ms of recorded time**. It is an *event-count image*, **not an ESCM output** and not a conventional intensity frame.

It uses the original event coordinates, without interpreting event polarity. This visualization does not alter the loaded arrays.

In [ ]:
import matplotlib.pyplot as plt

WINDOW_S = 0.050
mask = ts < (ts[0] + WINDOW_S) if len(ts) else np.zeros(0, dtype=bool)
count_image = np.zeros((HEIGHT, WIDTH), dtype=np.int32)
np.add.at(count_image, (y[mask], x[mask]), 1)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(count_image, cmap="magma", origin="upper")
ax.set(title=f"Recorded event counts, first {WINDOW_S * 1000:.0f} ms",
       xlabel="x (pixel)", ylabel="y (pixel)")
fig.colorbar(im, ax=ax, label="Events per pixel")
plt.show()

## 2. Next stage: mapping recorded events to model input

**Not implemented yet.** We will introduce a separate adapter rather than adding model behavior to the loader.

Questions to resolve in this stage:
- How do we convert `(x, y)` to a per-pixel input index (historically `i = y × width + x`)?
- How are continuous event timestamps represented relative to a numerical simulation clock?
- What happens if several events target the same pixel within one integration step?
- Should a change of recording or decoder ever change the model's time base? (**No—this must be explicit.**)

The definition of what an input event **does** to a neuron remains outside the loader and will be addressed in the neuron/input-dynamics stage.

We will add runnable cells here only after deciding the adapter's contract.

## 3. Planned neuron, persistence, interaction, and output stages

**No executable implementation in these sections yet.** Each stage will be developed and documented alongside focused tests:

**Neuron dynamics.** State variables, input coupling, integration rule, threshold, and reset. We will document the equations and their numerical implementation together.

**Persistence.** Define the state transitions that sustain activity after the event camera becomes quiet. Transition timing must be explicit rather than an accidental consequence of Brian's scheduling.

**Recurrent interactions.** Define self-excitation and inhibitory neighborhood geometry, including how edge pixels and simultaneous spikes behave.

**Output and validation.** Record a time-stamped **output spike stream**. Any frame images will be derived visualizations, not assumed to be the primary model output. We will compare isolated tests and recorded-data traces against the chosen reference.

### Working convention

For every new stage, this notebook will present: **purpose → scientific meaning → API → runnable example → basic checks → known limitations**. No component should be described as implemented until its code and tests exist.

The original Brian2 implementation remains available as historical evidence, but known implementation mistakes will not be silently inherited.